<a href="https://colab.research.google.com/github/solasobambo-prog/ts-data-science/blob/main/Week3c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Outliers
**IQR and Z-Score**

In [19]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 60

units = rng.integers(1, 21, size=n)
revenue = np.round(rng.lognormal(mean=6.4, sigma=0.55, size=n), 0)
units[17] = 4500
revenue[41] = 60000
revenue[8] = 9800
revenue[30] = 4200

sales = pd.DataFrame({
    'order_id': range(7001, 7001 + n),
    'region': rng.choice(['Lagos', 'Abuja', 'Kano', 'Ibadan'], size=n),
    'units': units,
    'revenue': revenue,
})

# Capture raw state to maintain restart-and-run-all discipline
sales_raw = sales.copy()


### Notes & Predictions: Part A (The IQR Rule)

* **Lower Fence Prediction:** The lower fence for revenue is predicted to be **negative**. Because revenue cannot physically fall below zero, a negative lower fence indicates that it is mathematically impossible to have low outliers in this specific dataset.
* **Revenue Outliers Prediction (IQR):** We predict **4 rows** will be flagged by the IQR rule. These include the 3 extreme values we planted manually (60,000, 9,800, and 4,200), alongside any naturally occurring extreme variations from the lognormal distribution.
* **Units Outliers Prediction (IQR):** We predict exactly **1 row** will be flagged (index 17, with 4,500 units). The rest of the values are bound tightly between 1 and 20 by the random generator.


In [20]:
print("=== PART A: THE IQR RULE ===")

# 1. Revenue Calculations
rev_q1 = sales['revenue'].quantile(0.25)
rev_q3 = sales['revenue'].quantile(0.75)
rev_iqr = rev_q3 - rev_q1
rev_lower = rev_q1 - 1.5 * rev_iqr
rev_upper = rev_q3 + 1.5 * rev_iqr

print(f"Revenue Metrics -> Q1: {rev_q1}, Q3: {rev_q3}, IQR: {rev_iqr}")
print(f"Revenue Fences  -> Lower: {rev_lower}, Upper: {rev_upper}")

rev_outliers_iqr = sales[(sales['revenue'] < rev_lower) | (sales['revenue'] > rev_upper)]
print(f"\nTotal Revenue Outliers (IQR): {len(rev_outliers_iqr)}")
display(rev_outliers_iqr)

# 2. Units Calculations
units_q1 = sales['units'].quantile(0.25)
units_q3 = sales['units'].quantile(0.75)
units_iqr = units_q3 - units_q1
units_lower = units_q1 - 1.5 * units_iqr
units_upper = units_q3 + 1.5 * units_iqr

print(f"\nUnits Metrics -> Q1: {units_q1}, Q3: {units_q3}, IQR: {units_iqr}")
print(f"Units Fences  -> Lower: {units_lower}, Upper: {units_upper}")

units_outliers_iqr = sales[(sales['units'] < units_lower) | (sales['units'] > units_upper)]
print(f"Total Units Outliers (IQR): {len(units_outliers_iqr)}")
display(units_outliers_iqr)


=== PART A: THE IQR RULE ===
Revenue Metrics -> Q1: 479.75, Q3: 854.25, IQR: 374.5
Revenue Fences  -> Lower: -82.0, Upper: 1416.0

Total Revenue Outliers (IQR): 4


,order_id,region,units,revenue
0,7001,Abuja,2,1955.0
8,7009,Kano,5,9800.0
30,7031,Kano,10,4200.0
41,7042,Abuja,16,60000.0



Units Metrics -> Q1: 8.0, Q3: 16.0, IQR: 8.0
Units Fences  -> Lower: -4.0, Upper: 28.0
Total Units Outliers (IQR): 1


,order_id,region,units,revenue
17,7018,Ibadan,4500,681.0


### Notes & Predictions: Part B (The Z-Score Rule)

* **Z-Score Filter Prediction:** The Z-score rule will flag **fewer rows** than the IQR rule.
* **The Reason:** Z-scores rely directly on the mean and standard deviation. Because our planted outlier of 60,000 is massive, it heavily inflates the column's standard deviation. This inflated denominator compresses the Z-scores of smaller anomalies (like 4,200 and 9,800), masking them and allowing only the absolute largest outlier to breach the `|Z| > 3` threshold.


In [21]:
print("=== PART B: THE Z-SCORE WITH UNITS CHECK ===")

# 1. Revenue Z-Scores
rev_mean = sales['revenue'].mean()
rev_std = sales['revenue'].std()
sales['rev_z'] = (sales['revenue'] - rev_mean) / rev_std
rev_outliers_z = sales[sales['rev_z'].abs() > 3]

print(f"Total Revenue Outliers (Z-score > 3): {len(rev_outliers_z)}")

# 2. Units Z-Scores (Fixing the missing 7018 check)
units_mean = sales['units'].mean()
units_std = sales['units'].std()
sales['units_z'] = (sales['units'] - units_mean) / units_std
units_outliers_z = sales[sales['units_z'].abs() > 3]

print(f"Total Units Outliers (Z-score > 3): {len(units_outliers_z)}")
display(sales.loc[sales['units_z'].abs() > 3, ['order_id', 'units', 'units_z']])


=== PART B: THE Z-SCORE WITH UNITS CHECK ===
Total Revenue Outliers (Z-score > 3): 1
Total Units Outliers (Z-score > 3): 1


,order_id,units,units_z
17,7018,4500,7.616532


In [22]:
# Programmatically build your matrix columns to avoid any transcription drifts
# Ensure the required helper columns and outlier flags exist in the current session state
all_outlier_ids = set(rev_outliers_iqr['order_id']).union(set(units_outliers_iqr['order_id']))
sales['is_outlier'] = sales['order_id'].isin(all_outlier_ids)

outlier_summary = sales[sales['is_outlier']].copy()

actions = {7001: 'FLAG', 7009: 'ASK', 7018: 'ASK', 7031: 'FLAG', 7042: 'CAP'}
justifications = {
    7001: 'High unit price, but entirely plausible for a premium product mix. Retain value.',
    7009: 'High unit value (~1,960/unit). Flag to verify if this represents a bulk premium combo.',
    7018: 'Clear Critical Error. 4,500 units for 681 total revenue is a data entry typo.',
    7031: 'Elevated revenue but realistic (~420/unit). Retain original value.',
    7042: 'Massive outlier distorting the pipeline mean. Cap to limit modeling leverage.'
}

outlier_summary['Action'] = outlier_summary['order_id'].map(actions)
outlier_summary['Justification'] = outlier_summary['order_id'].map(justifications)

# Complete 4-way flag tracking
outlier_summary['Rev IQR?'] = outlier_summary['order_id'].isin(rev_outliers_iqr['order_id']).map({True: '**Yes**', False: 'No'})
outlier_summary['Rev Z?'] = outlier_summary['order_id'].isin(rev_outliers_z['order_id']).map({True: '**Yes**', False: 'No'})
outlier_summary['Units IQR?'] = outlier_summary['order_id'].isin(units_outliers_iqr['order_id']).map({True: '**Yes**', False: 'No'})
outlier_summary['Units Z?'] = outlier_summary['order_id'].isin(units_outliers_z['order_id']).map({True: '**Yes**', False: 'No'})

print(outlier_summary[['order_id', 'region', 'units', 'revenue', 'Rev IQR?', 'Rev Z?', 'Units IQR?', 'Units Z?', 'Action', 'Justification']].to_markdown(index=False))

|   order_id | region   |   units |   revenue | Rev IQR?   | Rev Z?   | Units IQR?   | Units Z?   | Action   | Justification                                                                          |
|-----------:|:---------|--------:|----------:|:-----------|:---------|:-------------|:-----------|:---------|:---------------------------------------------------------------------------------------|
|       7001 | Abuja    |       2 |      1955 | **Yes**    | No       | No           | No         | FLAG     | High unit price, but entirely plausible for a premium product mix. Retain value.       |
|       7009 | Kano     |       5 |      9800 | **Yes**    | No       | No           | No         | ASK      | High unit value (~1,960/unit). Flag to verify if this represents a bulk premium combo. |
|       7018 | Ibadan   |    4500 |       681 | No         | No       | **Yes**      | **Yes**    | ASK      | Clear Critical Error. 4,500 units for 681 total revenue is a data entry typo.          |


### Outlier Justification and Action Matrix

|  Order ID | Region   |   Units |   Revenue | Rev IQR?   | Rev Z?   | Units IQR?   | Units Z?   | Action   | Justification                                                                |
|----------:|:---------|--------:|----------:|:-----------|:---------|:-------------|:-----------|:---------|:-----------------------------------------------------------------------------|
|      7001 | Abuja    |       2 |      1955 | **Yes**    | No       | No           | No         | FLAG     | High unit price, but entirely plausible for a premium product mix. Retain value. |
|      7009 | Kano     |       5 |      9800 | **Yes**    | No       | No           | No         | ASK      | High unit value (~1,960/unit). Flag to verify if this represents a bulk premium combo. |
|      7018 | Ibadan   |    4500 |       681 | No         | No       | **Yes**      | **Yes**    | ASK      | Clear Critical Error. 4,500 units for 681 total revenue is a data entry typo. |
|      7031 | Kano     |      10 |      4200 | **Yes**    | No       | No           | No         | FLAG     | Elevated revenue but realistic (~420/unit). Retain original value.          |
|      7042 | Abuja    |      16 |     60000 | **Yes**    | **Yes**  | No           | No         | CAP      | Massive outlier distorting the pipeline mean. Cap to limit modeling leverage. |

#### Data Owner Consultation Priority
We should immediately query the data owner regarding **Order 7018** to fix the unit-volume typo, and **Order 7042** to verify whether the 60,000 transaction represents a legitimate corporate contract or an accidental keystroke duplication.


In [23]:
print("=== PART C1: DECIDE AND TRANSFROM ===")

# 1. Flagging decisions for the combined outlier population
# Index 0   (Order 7001): FLAG. 1,955 revenue for 2 units is high but completely plausible.
# Index 8   (Order 7009): FLAG/ASK. 9,800 revenue for 5 units implies high luxury item mix; verify with owner.
# Index 30  (Order 7031): FLAG/ASK. 4,200 revenue for 10 units is elevated but realistic; verify pricing category.
# Index 41  (Order 7042): CAP. 60,000 revenue for 16 units is a massive outlier distorting pipeline means; cap it.
# Index 17  (Order 7018): ASK owner. 4,500 units sold with only 681 revenue is a massive data entry typo.

# 2. Build explicit transformation metrics without discarding entries
all_outlier_ids = set(rev_outliers_iqr['order_id']).union(set(units_outliers_iqr['order_id']))

sales['is_outlier'] = sales['order_id'].isin(all_outlier_ids)
sales['revenue_capped'] = sales['revenue'].clip(upper=rev_upper)

# 3. Summary Performance Reporting
print(f"Original Revenue -> Mean: {sales['revenue'].mean():.2f}, Median: {sales['revenue'].median():.2f}")
print(f"Capped Revenue   -> Mean: {sales['revenue_capped'].mean():.2f}, Median: {sales['revenue_capped'].median():.2f}")


=== PART C1: DECIDE AND TRANSFROM ===
Original Revenue -> Mean: 1874.92, Median: 657.50
Capped Revenue   -> Mean: 703.40, Median: 657.50


In [24]:
print("=== PART C: IMPLEMENT SPECIFIC DECISIONS ===")

# Isolate all outlier IDs across both rules for our boolean flag
all_outlier_ids = set(rev_outliers_iqr['order_id']).union(set(units_outliers_iqr['order_id']))
sales['is_outlier'] = sales['order_id'].isin(all_outlier_ids)

# Corrected Capping: Target ONLY order 7042, leaving 7001, 7009, and 7031 intact
sales['revenue_capped'] = sales['revenue']
mask = sales['order_id'] == 7042
sales.loc[mask, 'revenue_capped'] = rev_upper

# Print updated comparative metrics
print(f"Original Revenue -> Mean: {sales['revenue'].mean():.2f} | Median: {sales['revenue'].median():.2f}")
print(f"Capped Revenue   -> Mean: {sales['revenue_capped'].mean():.2f} | Median: {sales['revenue_capped'].median():.2f}")



=== PART C: IMPLEMENT SPECIFIC DECISIONS ===
Original Revenue -> Mean: 1874.92 | Median: 657.50
Capped Revenue   -> Mean: 898.52 | Median: 657.50


*   **Methodology Validation:** The shift in our capped mean from 703.40 to 898.52 shows the difference between global clipping and targeted capping. Global clipping over-corrected by dragging down all four rows, while our selective `.loc` mask successfully isolated and capped only the single 60,000 outlier, protecting the true operational value of our other premium transactions.


### Summary Analysis: Statistical Gap Interpretations

* **The Gap Interpretation (Original Data):** The large gap between the original mean (1,874.92) and the median (657.50) shows severe right-skewness, caused heavily by the leverage of the single 60,000 transaction.
* **The Gap Interpretation (Targeted Capping):** After capping *only* order 7042 at the upper fence value of **1,416**, the mean drops down to **898.52**, while the median remains steady at **657.50**.
* **Honest Distribution Reading:** The gap between the mean and median narrowed significantly from about 1,217 to about 241, but it remains positive. The mean is still roughly 37% above the median because the 9,800 and 4,200 orders remain fully intact alongside the lognormal distribution's natural right tail. This shows that our targeted capping successfully reduced the extreme skew without completely eliminating the distribution's inherent right tail.
* **Accounting and Financial Reporting Note:** Any official revenue or financial report sent to stakeholders must continue to use the raw `revenue` column, not `revenue_capped`. If the 60,000 order is a genuine corporate transaction, using the capped column would artificially understate true gross sales numbers. The capped column exists solely to stabilize model training.




In [25]:
print("=== PART D: CHALLENGE FUNCTION ===")

def iqr_outliers(df, col):
    """
    Computes IQR thresholds and prints upper/lower fences,
    returning a filtered DataFrame containing the flagged rows.
    """
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    print(f"\n--- Fences for Column [{col}] ---")
    print(f"Lower Fence: {lower_fence:.2f} | Upper Fence: {upper_fence:.2f}")

    flagged_df = df[(df[col] < lower_fence) | (df[col] > upper_fence)]
    print(f"Rows flagged as outliers: {len(flagged_df)}")
    return flagged_df

# Execute challenges on the unmutated tracking reference
outliers_units = iqr_outliers(sales_raw, 'units')
outliers_revenue = iqr_outliers(sales_raw, 'revenue')


=== PART D: CHALLENGE FUNCTION ===

--- Fences for Column [units] ---
Lower Fence: -4.00 | Upper Fence: 28.00
Rows flagged as outliers: 1

--- Fences for Column [revenue] ---
Lower Fence: -82.00 | Upper Fence: 1416.00
Rows flagged as outliers: 4
